In [1]:
import pandas as pd
import implicit
from sklearn.model_selection import train_test_split
import scipy.sparse as sparse
from pandas.api.types import CategoricalDtype 
import numpy as np
import warnings
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)

## Задание 1. Создание user-item-матрицы и разбиение данных на тест и контроль результатов

In [2]:
# Загрузка данных
ratings = pd.read_csv('ratings.csv')
ratings

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3
...,...,...,...
5976474,49925,510,5
5976475,49925,528,4
5976476,49925,722,4
5976477,49925,949,5


In [3]:
# Преобразование рейтингов в бинарные оценки
# Будем считать, если оценка больше 3, то книга понравилась
ratings['rating'] = ratings['rating'].apply(lambda x: 1 if x > 3 else 0)

# Пронумеруйте для каждого пользователя его прочитанные книги согласно расположению книг по порядку
ratings['book_order'] = ratings.groupby('user_id')['book_id'].rank(method='first')

# Переведите номера прочитанных книг в доли по формуле «порядковый номер / общее количество прочитанных пользователем книг»
ratings['book_order_ratio'] = ratings['book_order'] / ratings.groupby('user_id')['book_id'].transform('count')

ratings

,user_id,book_id,rating,book_order,book_order_ratio
0,1,258,1,63.0,0.538462
1,2,4081,1,54.0,0.830769
2,2,260,1,30.0,0.461538
3,2,9296,1,63.0,0.969231
4,2,2318,0,48.0,0.738462
...,...,...,...,...,...
5976474,49925,510,1,41.0,0.303704
5976475,49925,528,1,42.0,0.311111
5976476,49925,722,1,45.0,0.333333
5976477,49925,949,1,51.0,0.377778


In [4]:
# Разделите данные на обучение и контроль
train, test = train_test_split(ratings, train_size=0.8, random_state=42, shuffle=False)

In [5]:
# Построение user-item-матрицы
train_user_index = train.user_id.unique()
train_book_index = train.book_id.unique()
test_user_index = test.user_id.unique()
test_book_index = test.book_id.unique()

rows = train['user_id'].astype(pd.CategoricalDtype(categories=train_user_index)).cat.codes
cols = train['book_id'].astype(pd.CategoricalDtype(categories=train_book_index)).cat.codes
train_matrix = sparse.csr_matrix((train.rating, (rows, cols)), shape=(len(train_user_index), len(train_book_index)))

rows = test['user_id'].astype(pd.CategoricalDtype(categories=test_user_index)).cat.codes
cols = test['book_id'].astype(pd.CategoricalDtype(categories=test_book_index)).cat.codes
test_matrix = sparse.csr_matrix((test.rating, (rows, cols)), shape=(len(test_user_index), len(test_book_index)))


In [6]:
train_matrix = train_matrix.toarray()
test_matrix = test_matrix.toarray()

In [7]:
# Проверка размеров матриц
if train_matrix.shape[0] == len(train_user_index):
    print(f"Количество строк в обучающей матрице ({train_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(train_user_index)})")
else:
    print(f"Ошибка! Количество строк в обучающей матрице ({train_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(train_user_index)})")

if train_matrix.shape[1] == len(train_book_index):
    print(f"Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(train_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(train_book_index)})")

if test_matrix.shape[0] == len(test_user_index):
    print(f"Количество строк в тестовой матрице ({test_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(test_user_index)})")
else:
    print(f"Ошибка! Количество строк в тестовой матрице ({test_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(test_user_index)})")

if test_matrix.shape[1] == len(test_book_index):
    print(f"Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(test_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(test_book_index)})")


Количество строк в обучающей матрице (48769) РАВНО Количеству уникальных пользователей (48769)
Количество столбцов в обучающей матрице (9693) РАВНО Количеству уникальных книг (9693)
Количество строк в тестовой матрице (32281) РАВНО Количеству уникальных пользователей (32281)
Количество столбцов в тестовой матрице (9999) РАВНО Количеству уникальных книг (9999)


## Задание 2. Применение метода матричной факторизации и сбор признаков для контентной модели

In [8]:
# Для удобства возьмем 2-го пользователя из списка
user_index = 2

In [15]:
def ap_at_k(recommendations, true_positives, k=10):
    """
    Подсчитывает метрику AP@K.

    Args:
        recommendations: Список рекомендаций (book_id).
        true_positives: Список книг, положительно оценённых пользователем.
        k: Количество рекомендаций, которые учитываются при подсчёте метрики.

    Returns:
        Значение метрики AP@K.
    """
    
    # Отрезаем список рекомендаций до k элементов
    recommendations = recommendations[:k]
    
    # Создаем список бинарных меток (1 - книга в рекомендациях есть в списке положительно оцененных, 0 - нет)
    y_true = [1 if book_ind in true_positives else 0 for book_ind in recommendations]
    
    # Используем sklearn.metrics.average_precision_score для подсчета AP@K
    ap = average_precision_score(y_true, [1]*len(y_true)) 
    
    return ap

In [16]:
random_users_inds = np.random.choice(len(train_user_index), size=50, replace=False)

In [17]:
from tqdm import tqdm 

In [18]:
user_id = train_user_index[user_index]

In [19]:
# Создание модели ALS
als_model = implicit.als.AlternatingLeastSquares(factors=64, iterations=30)

als_model.fit(sparse.csr_matrix(train_matrix), show_progress=True)


  0%|          | 0/30 [00:00<?, ?it/s]

In [20]:
# Расчет mAP@10
als_map10 = []
for rand_user_ind in tqdm(random_users_inds):  # Используем tqdm для прогрессбара
    user_id = train_user_index[rand_user_ind]
    true_positives = test[test.user_id == user_id][test.rating == 1].book_id.tolist()

    if true_positives:
        recommendations = als_model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=20)
        recommended_book_ids = train_book_index[recommendations[0]]
        ap10 = ap_at_k(recommended_book_ids, true_positives, k=10)
        als_map10.append(ap10)

100%|██████████████████████████████████████████| 50/50 [01:08<00:00,  1.37s/it]


In [21]:
mean_als_ap_at_10 = np.mean(als_map10)
print(f"Среднее значение AP@10 для ALS: {mean_als_ap_at_10}")


Среднее значение AP@10 для ALS: 0.024000000000000004


### Задание 3. Применение метода матричной факторизации и улучшение параметров алгоритма факторизации

In [55]:
train_df = train.copy()
test_df = test.copy()

books = pd.read_csv('books.csv')
genres_df = pd.read_json('goodreads_book_genres_initial.json', lines=True)

genres_df = genres_df[genres_df.book_id.isin(books.goodreads_book_id)]
genres_df.columns = ['book_id', 'genres_dict']

all_genres = set()
for dict_genre in genres_df.genres_dict:
    for elem in list(dict_genre.keys()):
        all_genres.add(elem)

# Создаем столбцы для всех жанров в genres_df
for genre in all_genres:
    genres_df[genre] = 0

def simple_one_hot(genre_dict, genre):
    if genre in genre_dict:
        return 1
    return 0

# Применяем one-hot кодирование к каждому жанру
for genre in all_genres:
    genres_df[genre] = genres_df.apply(lambda row: simple_one_hot(row['genres_dict'], genre), axis=1)

# Объединение books и genre_df по book_id
books = books.merge(genres_df, on='book_id', how='left')

# Удаление ненужных столбцов
books = books.drop(['genres_dict'], axis=1) 

In [53]:
train_df = train.copy()
test_df = test.copy()

books = pd.read_csv('books.csv')
genres_df = pd.read_json('goodreads_book_genres_initial.json', lines=True)

genres_df = genres_df[genres_df.book_id.isin(books.goodreads_book_id)]
genres_df.columns = ['book_id', 'genres_dict']

all_genres = set()
for dict_genre in genres_df.genres_dict:
    for elem in list(dict_genre.keys()):
        all_genres.add(elem)
        
for genre in all_genres:
    genres_df[genre] = 0

def simple_one_hot(genre_dict, genre):
    if genre in genre_dict:
        return 1
    return 0

for genre in all_genres:
    genres_df[genre] = genres_df.apply(lambda df: simple_one_hot(df['genres_dict'], genre), axis=1)

# Объединение books и genre_df по book_id
books = books.merge(genres_df, on='book_id', how='left')

# Удаление ненужных столбцов
books = books.drop(['genres_dict'], axis=1) 

In [56]:
books

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,poetry,children,"mystery, thriller, crime",fiction,non-fiction,"fantasy, paranormal","comics, graphic","history, historical fiction, biography",young-adult,romance
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,7130616,7130616,7392860,19,441019455,9.780441e+12,Ilona Andrews,2010.0,Bayou Moon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,9997,208324,208324,1084709,19,067973371X,9.780680e+12,Robert A. Caro,1990.0,Means of Ascent,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9997,9998,77431,77431,2393986,60,039330762X,9.780393e+12,Patrick O'Brian,1977.0,The Mauritius Command,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
9998,9999,8565083,8565083,13433613,7,61711527,9.780062e+12,Peggy Orenstein,2011.0,Cinderella Ate My Daughter: Dispatches from th...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=100, stop_words=stopwords.words('english'))
X = vectorizer.fit_transform(books.title)

tf_idf_df = pd.DataFrame(X.toarray(), columns=vectorizer.vocabulary_)
tf_idf_df.columns = ['tf_' + item for item in tf_idf_df.columns] 

books = pd.concat([books, tf_idf_df], axis=1)

books = books.drop(columns=[
                             'best_book_id',
                             'work_id',
                             'books_count',
                             'isbn',
                             'isbn13',
                             'authors',
                             'original_publication_year',
                             'original_title',
                             'language_code',
                             'average_rating',
                             'work_ratings_count',
                             'work_text_reviews_count',
                             'ratings_1',
                             'ratings_2',
                             'ratings_3',
                             'ratings_4',
                             'ratings_5',
                             'image_url',
                             'small_image_url'])

train_df = train_df.merge(books, on='book_id', how='left')
test_df = test_df.merge(books, on='book_id', how='left')
books.columns = ['book_'+item if item in all_genres else item for item in list(books)]

users_profiles = train_df.groupby('user_id')[list(all_genres)].sum()

users_profiles.columns = ['user_'+name for name in list(users_profiles)]

train_df.columns = ['book_'+item if item in all_genres else item for item in list(train_df)]
test_df.columns = ['book_'+item if item in all_genres else item for item in list(test_df)]

# 1. Признак для пользователя (средний рейтинг)
user_avg_ratings = ratings.groupby('user_id')['rating'].mean().reset_index()
user_avg_ratings = user_avg_ratings.rename(columns={'rating': 'user_avg_rating'})
users_profiles = users_profiles.merge(user_avg_ratings, on='user_id', how='left')
train_df = train_df.merge(users_profiles, on='user_id', how='left')
test_df = test_df.merge(users_profiles, on='user_id', how='left')


# 2. Признак для книги (популярность)
books['book_popularity'] = np.log(books['ratings_count'] + 1) # +1 чтобы избежать логарифма от нуля
train_df = train_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')
test_df = test_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')
books = books.drop(columns=['ratings_count', 'title'])

In [49]:
cols_to_drop = [ 'user_id',
                 'book_id',
                 'rating',
                 'book_order',
                 'book_order_ratio',
                 'goodreads_book_id', 
                 'ratings_count',
                  'title']

In [35]:
from catboost import CatBoostClassifier, Pool

In [52]:
genres_df

,book_id,genres_dict,poetry,children,"mystery, thriller, crime",fiction,non-fiction,"fantasy, paranormal","comics, graphic","history, historical fiction, biography",young-adult,romance
3,6066819,"{'fiction': 555, 'romance': 23, 'mystery, thri...",0,0,1,1,0,0,0,0,0,1
15,89375,"{'non-fiction': 534, 'history, historical fict...",0,0,0,1,1,0,1,1,0,0
583,54270,"{'history, historical fiction, biography': 108...",0,0,0,0,1,0,1,1,0,0
807,38568,"{'fantasy, paranormal': 1907, 'romance': 1598,...",0,0,0,1,0,1,0,0,0,1
816,38562,"{'fantasy, paranormal': 1002, 'romance': 896, ...",0,0,1,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2359417,7663,"{'fiction': 409, 'mystery, thriller, crime': 4...",0,0,1,1,1,0,0,0,0,0
2359672,6280379,"{'fiction': 1254, 'history, historical fiction...",0,0,0,1,0,1,0,1,0,0
2360252,6871646,"{'children': 1036, 'fiction': 57, 'fantasy, pa...",1,1,0,1,0,1,0,0,0,0
2360258,7657484,"{'comics, graphic': 1535, 'fiction': 70, 'fant...",0,0,0,1,0,1,1,1,1,0


In [50]:
# Обучение модели
CB_model = CatBoostClassifier(
                            iterations=1000,
                            task_type="GPU",
                            use_best_model=True,
                            eval_metric='Accuracy',
                            loss_function='MultiClass',
                            early_stopping_rounds=10,
                            custom_metric='Precision'  
                        )

CB_model.fit(train_df.drop(columns=cols_to_drop), 
              train_df['rating'], 
              eval_set=(test_df.drop(columns=cols_to_drop), test_df['rating']),
              verbose=False,
              plot=True) 

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [ ]:
# --- Обучение ALS ---

In [ ]:
train_user_index = ratings.user_id.unique()
train_book_index = ratings.book_id.unique()

rows = ratings['user_id'].astype(pd.CategoricalDtype(categories=train_user_index)).cat.codes
cols = ratings['book_id'].astype(pd.CategoricalDtype(categories=train_book_index)).cat.codes
train_matrix = sparse.csr_matrix((ratings.rating, (rows, cols)), shape=(len(train_user_index), len(train_book_index)))

als_model = implicit.als.AlternatingLeastSquares(factors=64, iterations=30)
als_model.fit(sparse.csr_matrix(train_matrix), show_progress=True)

In [ ]:
# --- Получение рекомендаций от ALS ---
als_recommendations = []
for rand_user_ind in tqdm(random_users_inds):
    user_id = train_user_index[rand_user_ind]  # Получаем user_id по индексу из train_user_index
    recommendations = als_model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=30)  # Получаем рекомендации
    als_recommended_book_ids = train_book_index[recommendations[0]]  # Преобразуем индексы книг в реальные ID
    als_recommendations.append(als_recommended_book_ids)  # Сохраняем рекомендации в список

In [ ]:
common_columns_users = list(set(train_df.columns) & set(users_profiles.columns))
common_columns_users

In [ ]:
train_df

In [ ]:
common_columns_books = list(set(train_df.columns) & set(books.columns))
common_columns_books

In [ ]:
hybrid_map10 = []
for i, rand_user_ind in enumerate(tqdm(random_users_inds)):
    user_id = test_user_index[rand_user_ind]  # Получаем user_id по индексу из train_user_index
    true_positives = test_df[test_df.user_id == user_id][test_df.rating == 1].book_id.tolist()

    if true_positives:
        # Ранжирование с помощью CatBoost
        user_features = users_profiles[common_columns_users].iloc[user_id]  # Получение признаков пользователя
        candidate_features = books[books.original_book_id.isin(als_recommendations[i])][common_columns_books]  # Используем als_recommendations[i]

        # Переименование признаков для пользователя, чтобы CatBoost мог их правильно обработать
        user_features = user_features.rename(index={'user_young-adult': 'young-adult_user', 'user_mystery, thriller, crime': 'mystery, thriller, crime_user', 
                                                                'user_fiction': 'fiction_user', 'user_non-fiction': 'non-fiction_user', 'user_poetry': 'poetry_user',
                                                                'user_fantasy, paranormal': 'fantasy, paranormal_user', 'user_comics, graphic': 'comics, graphic_user', 
                                                                'user_history, historical fiction, biography': 'history, historical fiction, biography_user', 
                                                                'user_children': 'children_user', 'user_romance': 'romance_user'})

        # Создание features DataFrame
        features = pd.concat([user_features.to_frame().T] * len(candidate_features), ignore_index=True).join(
            candidate_features, lsuffix='user', rsuffix='book'
        )

        # Проверка порядка признаков и перестановка, если нужно
        features = features[train_df.drop(columns=cols_to_drop).columns]  # Устанавливаем порядок признаков, соответствующий train_df

        # Проверка на наличие пропущенных значений и заполнение их нулями
        features.fillna(0, inplace=True)

        predictions = CB_model.predict_proba(
            features
        )[:, 1]  # Получение вероятностей для класса 1 (положительный рейтинг)
        top_recommendations = als_recommendations[i][np.argsort(predictions)[::-1][:10]]  # Выбор топ-10 книг по ранжированию

        # Расчет mAP@10
        ap10 = ap_at_k(top_recommendations, true_positives, k=10)
        hybrid_map10.append(ap10)

mean_hybrid_ap_at_10 = np.mean(hybrid_map10)
print(f"Среднее значение AP@10 для гибридной модели: {mean_hybrid_ap_at_10}")

In [ ]:
users_profiles.loc[user_id]

In [ ]:
train_df

In [ ]:
['book_fiction', 'book_non-fiction', 'book_history, historical fiction, biography', 'book_mystery, thriller, crime', 'book_romance', 'book_children', 'book_fantasy, paranormal', 'book_young-adult', 'book_poetry', 'book_comics, graphic', 'user_fiction', 'user_non-fiction', 'user_history, historical fiction, biography', 'user_mystery, thriller, crime', 'user_romance', 'user_children', 'user_fantasy, paranormal', 'user_young-adult', 'user_poetry', 'user_comics, graphic', 'user_avg_rating']